# PyTorch自定义模型与训练算法

> 本notebook是 [TensorFlow版本](./定制模型和训练算法.ipynb) 的PyTorch等价实现。

本notebook深入讲解PyTorch的高级自定义功能，包括自定义损失函数、层、模型和训练循环。这些技术是从"调包侠"进阶到"算法工程师"的关键能力。

## 学习目标
1. 掌握自定义损失函数的实现方法
2. 学会自定义层（Module）的创建
3. 理解自定义模型的构建
4. 实现自定义训练循环
5. 掌握模型的保存与加载

## 1. 环境设置与数据准备

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 设置随机种子 / Set random seed
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# 检测设备 / Detect device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch版本 / PyTorch version: {torch.__version__}")
print(f"设备 / Device: {device}")

In [ ]:
# 加载加州房价数据集 / Load California Housing dataset
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

housing = fetch_california_housing()
X_full = housing.data.astype(np.float32)
y_full = housing.target.astype(np.float32)

# 标准化特征 / Standardize features
scaler = StandardScaler()
X_full = scaler.fit_transform(X_full)

# 划分数据集 / Split dataset
X_train, X_temp, y_train, y_temp = train_test_split(
    X_full, y_full, train_size=0.64, random_state=RANDOM_SEED
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, train_size=0.5, random_state=RANDOM_SEED
)

# 转为PyTorch张量 / Convert to PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_valid_t = torch.tensor(X_valid, dtype=torch.float32)
y_valid_t = torch.tensor(y_valid, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# 创建DataLoader / Create DataLoaders
train_dataset = TensorDataset(X_train_t, y_train_t)
valid_dataset = TensorDataset(X_valid_t, y_valid_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"训练集 / Train: {X_train.shape[0]} 样本")
print(f"验证集 / Valid: {X_valid.shape[0]} 样本")
print(f"测试集 / Test: {X_test.shape[0]} 样本")
print(f"特征数 / Features: {X_train.shape[1]}")

## 2. 自定义损失函数

### 2.1 函数式损失函数

最简单的自定义损失函数是一个接受`y_pred`和`y_true`两个参数的函数。

**Huber损失函数**是一种结合MSE和MAE优点的鲁棒损失函数：
- 当误差较小时（|error| < δ），使用MSE（对小误差敏感）
- 当误差较大时（|error| ≥ δ），使用MAE（对异常值鲁棒）

> **TF vs PyTorch**: TF中损失函数签名为 `(y_true, y_pred)`，PyTorch中习惯为 `(y_pred, y_true)` 或 `(input, target)`，与 `nn.Module` 的 `forward` 调用顺序一致。PyTorch也内置了 `nn.HuberLoss`。

In [ ]:
def huber_fn(y_pred, y_true, delta=1.0):
    """
    Huber损失函数实现 / Huber loss function implementation

    参数 / Args:
        y_pred: 预测值 / predicted values
        y_true: 真实标签 / ground truth labels
        delta: 阈值δ / threshold δ

    返回 / Returns:
        损失值张量 / loss tensor

    数学公式 / Formula:
        L(y, f(x)) =
            0.5 * (y - f(x))^2,           if |y - f(x)| < δ
            δ * |y - f(x)| - 0.5 * δ^2,   otherwise
    """
    error = y_true - y_pred
    abs_error = torch.abs(error)
    quadratic = 0.5 * error ** 2
    linear = delta * abs_error - 0.5 * delta ** 2
    return torch.where(abs_error < delta, quadratic, linear)

# 测试Huber损失函数 / Test Huber loss function
y_true_test = torch.tensor([1.0, 2.0, 3.0])
y_pred_test = torch.tensor([1.5, 2.5, 5.0])  # 误差分别为0.5, 0.5, 2.0
loss = huber_fn(y_pred_test, y_true_test)
print(f"测试Huber损失 / Test Huber loss: {loss.numpy()}")
print("  误差0.5 -> 使用MSE: 0.5^2/2 = 0.125")
print("  误差2.0 -> 使用MAE: |2.0| - 0.5 = 1.5")

# 对比PyTorch内置Huber损失 / Compare with PyTorch built-in Huber loss
built_in_huber = nn.HuberLoss(delta=1.0, reduction='none')
builtin_loss = built_in_huber(y_pred_test.unsqueeze(0), y_true_test.unsqueeze(0))
print(f"\nPyTorch内置Huber损失 / Built-in Huber loss: {builtin_loss.squeeze().numpy()}")
print(f"结果一致 / Results match: {torch.allclose(loss, builtin_loss.squeeze(), atol=1e-5)}")

In [ ]:
# 使用自定义损失函数训练模型 / Train model with custom loss function

def build_regression_model(input_dim):
    """构建用于回归任务的MLP模型 / Build MLP model for regression"""
    return nn.Sequential(
        nn.Linear(input_dim, 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 1)  # 回归任务不需要激活函数 / No activation for regression
    )

model = build_regression_model(input_dim=X_train.shape[1]).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 训练循环 / Training loop
for epoch in range(5):
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        y_pred = model(X_batch)
        loss = huber_fn(y_pred, y_batch).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * X_batch.size(0)

    # 验证 / Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in valid_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            val_loss += huber_fn(y_pred, y_batch).mean().item() * X_batch.size(0)

    epoch_loss /= len(train_dataset)
    val_loss /= len(valid_dataset)
    print(f"Epoch {epoch+1}/5 - loss: {epoch_loss:.4f} - val_loss: {val_loss:.4f}")

# 评估 / Evaluate
model.eval()
test_loss = 0.0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = model(X_batch)
        test_loss += huber_fn(y_pred, y_batch).mean().item() * X_batch.size(0)
test_loss /= len(test_dataset)
print(f"\n测试集损失 / Test loss: {test_loss:.4f}")

### 2.2 带参数的损失函数（nn.Module子类）

如果损失函数需要可配置的参数（如Huber的阈值），可以继承`nn.Module`来实现。

> **TF vs PyTorch**: TF中通过继承`keras.losses.Loss`并实现`call()`方法；PyTorch中继承`nn.Module`并实现`forward()`方法。PyTorch的方式更统一——损失函数、层、模型都是`nn.Module`。

In [ ]:
class HuberLoss(nn.Module):
    """
    Huber损失函数类实现 / Huber loss class implementation

    继承nn.Module使参数可以随模型保存 / Subclassing nn.Module allows parameters to be saved with model

    参数 / Args:
        threshold: 阈值δ / threshold δ
        reduction: 归约方式 ('mean'或'none') / reduction method
    """

    def __init__(self, threshold=1.0, reduction='mean'):
        super().__init__()
        self.threshold = threshold
        self.reduction = reduction

    def forward(self, y_pred, y_true):
        """计算损失值 / Compute loss value"""
        error = y_true - y_pred
        abs_error = torch.abs(error)
        quadratic = 0.5 * error ** 2
        linear = self.threshold * abs_error - 0.5 * self.threshold ** 2
        loss = torch.where(abs_error < self.threshold, quadratic, linear)

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

# 测试损失类 / Test loss class
huber_loss_1 = HuberLoss(threshold=1.0, reduction='none')
huber_loss_2 = HuberLoss(threshold=2.0, reduction='none')

error_values = torch.tensor([0.5, 1.5, 3.0])
y_true = torch.tensor([0.0, 0.0, 0.0])
y_pred = -error_values

print(f"误差值 / Error values: {error_values.numpy()}")
print(f"阈值1.0的损失 / Loss with threshold=1.0: {huber_loss_1(y_pred.unsqueeze(0), y_true.unsqueeze(0)).squeeze().numpy()}")
print(f"阈值2.0的损失 / Loss with threshold=2.0: {huber_loss_2(y_pred.unsqueeze(0), y_true.unsqueeze(0)).squeeze().numpy()}")

# 使用损失类编译模型 / Use loss class with model
model = build_regression_model(input_dim=X_train.shape[1]).to(device)
criterion = HuberLoss(threshold=2.0)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
print("\n模型配置成功，使用HuberLoss(threshold=2.0) / Model configured with HuberLoss(threshold=2.0)")

## 3. 自定义激活函数、初始化器和正则化器

这些组件都可以通过简单的函数或类来自定义。

> **TF vs PyTorch**: PyTorch的初始化器通过`torch.nn.init`模块提供，使用方式是函数调用而非类实例。正则化器PyTorch没有内置，通常在损失函数中手动添加L1/L2项。

In [ ]:
# 自定义激活函数 - Softplus的手动实现 / Custom activation - manual Softplus
def my_softplus(z):
    """Softplus激活: log(exp(z) + 1)，是ReLU的平滑近似 / Softplus: smooth approximation of ReLU"""
    return torch.log(torch.exp(z) + 1.0)

# 自定义初始化器 - Glorot/Xavier初始化 / Custom initializer - Glorot/Xavier init
def my_glorot_initializer(tensor):
    """
    Glorot/Xavier均匀初始化 / Glorot/Xavier uniform initialization

    PyTorch中初始化器直接修改tensor（in-place），
    而TF中初始化器返回一个张量。
    """
    nn.init.xavier_uniform_(tensor)

# 自定义L1正则化器 / Custom L1 regularizer
def l1_regularizer(weights, factor=0.001):
    """
    L1正则化 / L1 regularization

    参数 / Args:
        weights: 权重张量 / weight tensor
        factor: 正则化系数 / regularization factor
    """
    return factor * torch.sum(torch.abs(weights))

# 使用自定义组件创建层 / Create layer with custom components
custom_layer = nn.Linear(5, 10)

# 应用自定义初始化 / Apply custom initialization
my_glorot_initializer(custom_layer.weight)

# 测试自定义层 / Test custom layer
test_input = torch.randn(2, 5)
output = my_softplus(custom_layer(test_input))
print(f"输入形状 / Input shape: {test_input.shape}")
print(f"输出形状 / Output shape: {output.shape}")

# 计算L1正则化损失 / Compute L1 regularization loss
reg_loss = l1_regularizer(custom_layer.weight, factor=0.01)
print(f"L1正则化损失 / L1 reg loss: {reg_loss.item():.4f}")

# 非负约束：在训练后裁剪权重 / Non-neg constraint: clip weights after training
with torch.no_grad():
    custom_layer.weight.clamp_(min=0)
print(f"权重是否非负 / Weights non-negative: {(custom_layer.weight >= 0).all().item()}")

## 4. 自定义评估指标

PyTorch没有像Keras那样的内置指标系统，但实现流式指标非常简单。

> **TF vs PyTorch**: TF中通过继承`keras.metrics.Metric`并实现`update_state()`/`result()`/`reset_state()`；PyTorch中只需一个简单的Python类，更加轻量。

In [ ]:
class HuberMetric:
    """
    流式Huber指标 / Streaming Huber metric

    在整个epoch中累积计算平均Huber损失 / Accumulates average Huber loss across epoch

    参数 / Args:
        threshold: 阈值δ / threshold δ
    """

    def __init__(self, threshold=1.0, name='huber_metric'):
        self.threshold = threshold
        self.name = name
        self.reset()

    def reset(self):
        """重置状态（每个epoch开始时调用） / Reset state (called at start of each epoch)"""
        self.total = 0.0
        self.count = 0

    def update(self, y_pred, y_true):
        """更新状态变量 / Update state variables"""
        with torch.no_grad():
            loss = huber_fn(y_pred, y_true, delta=self.threshold)
            self.total += loss.sum().item()
            self.count += loss.numel()

    def compute(self):
        """返回当前指标值 / Return current metric value"""
        if self.count == 0:
            return 0.0
        return self.total / self.count

# 测试自定义指标 / Test custom metric
metric = HuberMetric(threshold=1.0)

# 模拟多个批次的更新 / Simulate multi-batch updates
metric.update(torch.tensor([[1.5]]), torch.tensor([[1.0]]))
metric.update(torch.tensor([[2.2]]), torch.tensor([[2.0]]))
metric.update(torch.tensor([[3.1]]), torch.tensor([[3.0]]))
metric.update(torch.tensor([[5.5]]), torch.tensor([[4.0]]))

print(f"累积Huber指标 / Accumulated Huber metric: {metric.compute():.4f}")

# 重置 / Reset
metric.reset()
print(f"重置后 / After reset: {metric.compute():.4f}")

In [ ]:
# 更通用的指标类 / More general metric class

class MeanMetric:
    """
    通用均值指标 / General mean metric

    可用于追踪任意标量的均值 / Can track mean of any scalar
    """

    def __init__(self, name='mean'):
        self.name = name
        self.reset()

    def reset(self):
        self.total = 0.0
        self.count = 0

    def update(self, value, n=1):
        """更新指标 / Update metric with value and sample count"""
        self.total += value * n
        self.count += n

    def compute(self):
        return self.total / self.count if self.count > 0 else 0.0

# 测试 / Test
mean_metric = MeanMetric(name='avg_loss')
mean_metric.update(0.5, n=32)
mean_metric.update(0.3, n=32)
print(f"{mean_metric.name}: {mean_metric.compute():.4f}")

## 5. 自定义层

### 5.1 无权重的自定义层

In [ ]:
class GaussianNoise(nn.Module):
    """
    高斯噪声层 / Gaussian noise layer

    在训练时添加高斯噪声作为正则化手段，推理时不添加噪声
    Adds Gaussian noise during training as regularization; no noise during inference

    参数 / Args:
        stddev: 噪声标准差 / noise standard deviation
    """

    def __init__(self, stddev):
        super().__init__()
        self.stddev = stddev

    def forward(self, inputs):
        if self.training:  # self.training 是nn.Module内置属性 / built-in attribute of nn.Module
            noise = torch.randn_like(inputs) * self.stddev
            return inputs + noise
        return inputs

# 测试噪声层 / Test noise layer
noise_layer = GaussianNoise(stddev=0.1)
test_input = torch.tensor([[1.0, 2.0, 3.0]])

noise_layer.train()  # 训练模式 / Training mode
noisy_output = noise_layer(test_input)
print(f"原始输入 / Original input: {test_input.numpy()}")
print(f"训练模式输出 / Training mode output: {noisy_output.numpy()}")

noise_layer.eval()  # 推理模式 / Eval mode
eval_output = noise_layer(test_input)
print(f"推理模式输出 / Eval mode output: {eval_output.numpy()}")

# 对比TF: TF中通过call(inputs, training=None)参数控制，
# PyTorch中通过self.training属性自动管理（由model.train()/eval()设置）

### 5.2 带权重的自定义层

> **TF vs PyTorch**: TF中通过`build()`方法延迟创建权重；PyTorch中通过`__init__`中定义`nn.Parameter`或子模块，框架自动延迟初始化。PyTorch的权重管理更简洁——不需要手动调用`build()`。

In [ ]:
class MyDense(nn.Module):
    """
    自定义全连接层 / Custom fully-connected layer

    手动实现Dense/Linear层以理解权重管理机制
    Manual implementation of Dense/Linear layer to understand weight management

    参数 / Args:
        in_features: 输入特征数 / number of input features
        out_features: 输出特征数 / number of output features
        activation: 激活函数 / activation function
    """

    def __init__(self, in_features, out_features, activation=None):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.activation = activation

        # 定义可训练参数 / Define trainable parameters
        # nn.Parameter会自动注册到模块的parameters()中 / Auto-registers in parameters()
        self.weight = nn.Parameter(
            torch.empty(in_features, out_features)
        )
        self.bias = nn.Parameter(
            torch.empty(out_features)
        )

        # 初始化权重 / Initialize weights
        nn.init.xavier_uniform_(self.weight)
        nn.init.zeros_(self.bias)

    def forward(self, inputs):
        """前向传播 / Forward pass"""
        z = inputs @ self.weight + self.bias
        if self.activation is not None:
            return self.activation(z)
        return z

    def extra_repr(self):
        """打印模块信息 / Print module info (equivalent to TF's get_config)"""
        return f"in_features={self.in_features}, out_features={self.out_features}, activation={self.activation}"

# 测试自定义Dense层 / Test custom Dense layer
my_dense = MyDense(4, 5, activation=torch.relu)
test_input = torch.randn(3, 4)
output = my_dense(test_input)

print(f"输入形状 / Input shape: {test_input.shape}")
print(f"输出形状 / Output shape: {output.shape}")
print(f"权重形状 / Weight shape: {my_dense.weight.shape}")
print(f"偏置形状 / Bias shape: {my_dense.bias.shape}")
print(f"\n模块信息 / Module info:\n{my_dense}")

### 5.3 多输入/多输出层

In [ ]:
class MultiOutputLayer(nn.Module):
    """
    多输出计算层 / Multi-output computation layer

    接收两个输入，返回它们的和、积、商
    Takes two inputs, returns their sum, product, and quotient
    """

    def forward(self, x1, x2):
        return (
            x1 + x2,
            x1 * x2,
            x1 / (x2 + 1e-7)  # 添加小值避免除零 / Add small value to avoid division by zero
        )

# 测试多输出层 / Test multi-output layer
multi_layer = MultiOutputLayer()
x1 = torch.tensor([[1.0, 2.0]])
x2 = torch.tensor([[3.0, 4.0]])
sum_out, prod_out, quot_out = multi_layer(x1, x2)

print(f"输入x1 / Input x1: {x1.numpy()}")
print(f"输入x2 / Input x2: {x2.numpy()}")
print(f"和 / Sum: {sum_out.numpy()}")
print(f"积 / Product: {prod_out.numpy()}")
print(f"商 / Quotient: {quot_out.numpy()}")

## 6. 自定义模型

### 6.1 残差块和残差网络

> **TF vs PyTorch**: TF中继承`keras.Model`实现`call()`；PyTorch中继承`nn.Module`实现`forward()`。两者结构非常相似，但PyTorch不需要`build()`方法——参数在`__init__`中定义即可。

In [ ]:
class ResidualBlock(nn.Module):
    """
    残差块 / Residual block

    实现跳跃连接：output = F(x) + x
    Implements skip connection: output = F(x) + x

    参数 / Args:
        n_layers: 隐藏层数量 / number of hidden layers
        n_neurons: 每层神经元数 / neurons per layer
        in_features: 输入特征数 / input features (needed for PyTorch)
    """

    def __init__(self, n_layers, n_neurons, in_features):
        super().__init__()
        self.hidden = nn.ModuleList([
            nn.Linear(in_features if i == 0 else n_neurons, n_neurons)
            for i in range(n_layers)
        ])
        self.activation = nn.ELU()
        # 如果输入输出维度不同，需要投影层 / Projection layer if dims differ
        self.projection = None

    def forward(self, inputs):
        z = inputs
        for layer in self.hidden:
            z = self.activation(layer(z))
        # 残差连接 / Residual connection
        if self.projection is not None:
            return self.projection(inputs) + z
        return inputs + z


class ResidualRegressor(nn.Module):
    """
    残差回归模型 / Residual regression model

    使用残差块构建的回归网络 / Regression network built with residual blocks

    参数 / Args:
        input_dim: 输入特征数 / input dimension
        output_dim: 输出维度 / output dimension
    """

    def __init__(self, input_dim, output_dim=1):
        super().__init__()
        self.hidden1 = nn.Linear(input_dim, 30)
        self.activation1 = nn.ELU()
        self.block1 = ResidualBlock(2, 30, in_features=30)
        self.block2 = ResidualBlock(2, 30, in_features=30)
        self.out = nn.Linear(30, output_dim)

    def forward(self, inputs):
        z = self.activation1(self.hidden1(inputs))
        z = self.block1(z)
        z = self.block2(z)
        return self.out(z)

# 测试残差模型 / Test residual model
res_model = ResidualRegressor(input_dim=8, output_dim=1).to(device)
test_input = torch.randn(5, 8).to(device)
output = res_model(test_input)

print(f"输入形状 / Input shape: {test_input.shape}")
print(f"输出形状 / Output shape: {output.shape}")
print(f"\n模型结构 / Model structure:\n{res_model}")

### 6.2 带辅助损失的模型

> **TF vs PyTorch**: TF中使用`self.add_loss()`自动将辅助损失加入总损失；PyTorch中需要在训练循环中手动计算并合并辅助损失。PyTorch的方式更透明——你完全掌控损失的计算。

In [ ]:
class ReconstructingRegressor(nn.Module):
    """
    带重建损失的回归模型 / Regression model with reconstruction loss

    除了主要的回归任务外，还添加输入重建作为辅助任务，
    这有助于学习更好的特征表示

    Besides the main regression task, adds input reconstruction as auxiliary task,
    which helps learn better feature representations

    参数 / Args:
        input_dim: 输入特征数 / input dimension
        output_dim: 输出维度 / output dimension
        recon_weight: 重建损失权重 / reconstruction loss weight
    """

    def __init__(self, input_dim, output_dim=1, recon_weight=0.05):
        super().__init__()
        self.recon_weight = recon_weight
        self.hidden = nn.ModuleList([
            nn.Linear(input_dim if i == 0 else 30, 30)
            for i in range(3)
        ])
        self.activation = nn.SELU()
        self.out = nn.Linear(30, output_dim)
        self.reconstruct = nn.Linear(30, input_dim)

    def forward(self, inputs):
        """
        前向传播 / Forward pass

        返回 / Returns:
            (prediction, reconstruction_loss): 预测值和重建损失
        """
        z = inputs
        for layer in self.hidden:
            z = self.activation(layer(z))

        # 计算重建损失 / Compute reconstruction loss
        reconstruction = self.reconstruct(z)
        recon_loss = nn.functional.mse_loss(reconstruction, inputs)

        return self.out(z), recon_loss

# 训练带辅助损失的模型 / Train model with auxiliary loss
recon_model = ReconstructingRegressor(input_dim=X_train.shape[1]).to(device)
optimizer = optim.Adam(recon_model.parameters(), lr=1e-3)
main_criterion = nn.MSELoss()

for epoch in range(3):
    recon_model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        y_pred, recon_loss = recon_model(X_batch)
        main_loss = main_criterion(y_pred, y_batch)
        # 合并主损失和辅助损失 / Combine main loss and auxiliary loss
        total_loss = main_loss + recon_model.recon_weight * recon_loss

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        epoch_loss += total_loss.item() * X_batch.size(0)

    epoch_loss /= len(train_dataset)
    print(f"Epoch {epoch+1}/3 - total_loss: {epoch_loss:.4f} (main + 0.05*recon)")

print("\n模型总损失包含主损失+辅助损失 / Total loss includes main + auxiliary loss")

## 7. 自定义训练循环

当标准的训练循环无法满足需求时（如GAN训练、强化学习），需要自定义训练循环。

### PyTorch训练循环 vs TF GradientTape

**这是PyTorch相比TF最自然的部分！**

| 步骤 | TensorFlow (GradientTape) | PyTorch |
|------|--------------------------|---------|
| 1. 前向传播 | `with tf.GradientTape() as tape: y_pred = model(X)` | `y_pred = model(X)` |
| 2. 计算损失 | `loss = loss_fn(y_true, y_pred)` | `loss = loss_fn(y_pred, y_true)` |
| 3. 计算梯度 | `grads = tape.gradient(loss, vars)` | `loss.backward()` |
| 4. 更新参数 | `optimizer.apply_gradients(zip(grads, vars))` | `optimizer.step()` |
| 5. 清零梯度 | *(自动)* | `optimizer.zero_grad()` |

PyTorch的优势：
- **无需GradientTape上下文管理器** — `loss.backward()`自动计算梯度
- **更少的样板代码** — 3行代码完成梯度计算和参数更新
- **更直观** — 前向传播就是普通函数调用，不需要包裹在`with`语句中

In [ ]:
def print_status_bar(step, total, loss, metrics=None):
    """打印训练进度条 / Print training progress bar"""
    metrics_str = " - ".join(
        [f"{m.name}: {m.compute():.4f}" for m in (metrics or [])]
    )
    end = "" if step < total else "\n"
    print(f"\r{step}/{total} - loss: {loss:.4f} - {metrics_str}", end=end)

In [ ]:
# 创建模型和优化器 / Create model and optimizer
model = nn.Sequential(
    nn.Linear(X_train.shape[1], 30),
    nn.ELU(),
    nn.Linear(30, 1)
).to(device)

# 超参数 / Hyperparameters
n_epochs = 3
batch_size = 32
n_steps = len(X_train) // batch_size

# 优化器和损失函数 / Optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# 指标 / Metrics
mean_loss = MeanMetric(name='loss')
mae_metric = MeanMetric(name='mae')

# L2正则化系数 / L2 regularization factor
l2_factor = 0.001

# 自定义训练循环 / Custom training loop
print("开始自定义训练循环... / Starting custom training loop...")
for epoch in range(n_epochs):
    print(f"\nEpoch {epoch + 1}/{n_epochs}")

    # 重置指标 / Reset metrics
    mean_loss.reset()
    mae_metric.reset()

    model.train()
    for step in range(n_steps):
        # 获取批次数据 / Get batch data
        start = step * batch_size
        end = start + batch_size
        X_batch = X_train_t[start:end].to(device)
        y_batch = y_train_t[start:end].to(device)

        # ===== PyTorch训练循环核心（3行代码）=====
        # ===== Core PyTorch training loop (3 lines) =====
        y_pred = model(X_batch)                          # 前向传播 / Forward pass
        main_loss = loss_fn(y_pred, y_batch)             # 计算损失 / Compute loss
        # 加入L2正则化损失 / Add L2 regularization loss
        l2_loss = l2_factor * sum(p.pow(2).sum() for p in model.parameters())
        total_loss = main_loss + l2_loss

        optimizer.zero_grad()                            # 清零梯度 / Zero gradients
        total_loss.backward()                            # 反向传播 / Backward pass
        optimizer.step()                                 # 更新参数 / Update parameters
        # ===== 对比TF需要5行+GradientTape =====
        # ===== Compare: TF needs 5 lines + GradientTape =====
        # with tf.GradientTape() as tape:
        #     y_pred = model(X_batch, training=True)
        #     loss = loss_fn(y_batch, y_pred) + sum(model.losses)
        # grads = tape.gradient(loss, model.trainable_variables)
        # optimizer.apply_gradients(zip(grads, model.trainable_variables))

        # 更新指标 / Update metrics
        mean_loss.update(total_loss.item(), n=X_batch.size(0))
        with torch.no_grad():
            mae = torch.abs(y_pred - y_batch).mean().item()
        mae_metric.update(mae, n=X_batch.size(0))

        # 每100步打印一次 / Print every 100 steps
        if step % 100 == 0:
            print_status_bar(step, n_steps, mean_loss.compute(), [mae_metric])

    # epoch结束打印 / Print at end of epoch
    print_status_bar(n_steps, n_steps, mean_loss.compute(), [mae_metric])

print("\n训练完成! / Training complete!")

## 8. 保存和加载自定义组件

> **TF vs PyTorch**: TF使用`model.save()`保存整个模型（包括架构），加载时需要`custom_objects`；PyTorch使用`torch.save()`保存`state_dict`（仅权重），加载时需要先创建模型实例。PyTorch的方式更灵活——架构和权重分离保存。

In [ ]:
import os
import tempfile

# 创建使用自定义组件的模型 / Create model with custom components
model = nn.Sequential(
    nn.Linear(X_train.shape[1], 30),
    nn.ReLU(),
    nn.Linear(30, 1)
).to(device)

# 使用自定义损失训练一个epoch / Train one epoch with custom loss
criterion = HuberLoss(threshold=1.5)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

model.train()
for X_batch, y_batch in train_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    y_pred = model(X_batch)
    loss = criterion(y_pred, y_batch)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# 保存模型 / Save model
with tempfile.TemporaryDirectory() as tmpdir:
    model_path = os.path.join(tmpdir, 'model_with_custom_loss.pt')
    # PyTorch保存state_dict（推荐方式）/ Save state_dict (recommended)
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'huber_threshold': 1.5,  # 保存自定义损失参数 / Save custom loss params
    }, model_path)
    print(f"模型已保存到 / Model saved to: {model_path}")

    # 加载模型 / Load model
    # 需要先创建模型实例（架构代码必须可用）/ Need to create model instance first
    loaded_model = nn.Sequential(
        nn.Linear(X_train.shape[1], 30),
        nn.ReLU(),
        nn.Linear(30, 1)
    ).to(device)

    checkpoint = torch.load(model_path, weights_only=False)
    loaded_model.load_state_dict(checkpoint['model_state_dict'])
    loaded_criterion = HuberLoss(threshold=checkpoint['huber_threshold'])
    print("模型加载成功! / Model loaded successfully!")
    print(f"恢复Huber阈值 / Restored Huber threshold: {checkpoint['huber_threshold']}")

    # 验证 / Verify
    model.eval()
    loaded_model.eval()
    with torch.no_grad():
        X_sample = X_test_t[:5].to(device)
        original_pred = model(X_sample)
        loaded_pred = loaded_model(X_sample)
        print(f"预测结果一致 / Predictions match: {torch.allclose(original_pred, loaded_pred, atol=1e-5)}")

## TF vs PyTorch 对照

### 自定义组件对照表

| 组件 | TensorFlow | PyTorch | 差异说明 |
|------|-----------|---------|----------|
| **损失函数** | `keras.losses.Loss` 子类，实现 `call()` | `nn.Module` 子类，实现 `forward()`，或简单函数 | PyTorch签名 `(y_pred, y_true)`，TF签名 `(y_true, y_pred)` |
| **层** | `keras.layers.Layer` 子类，实现 `build()` + `call()` | `nn.Module` 子类，实现 `__init__()` + `forward()` | PyTorch不需要`build()`，参数在`__init__`中定义 |
| **模型** | `keras.Model` 子类，实现 `call()` | `nn.Module` 子类，实现 `forward()` | 几乎相同，PyTorch用`forward`而非`call` |
| **指标** | `keras.metrics.Metric` 子类 | 简单Python类 | PyTorch更轻量，无框架约束 |
| **训练循环** | `tf.GradientTape` + `apply_gradients` | `loss.backward()` + `optimizer.step()` | PyTorch更自然，3行代码 vs 5行 |
| **保存/加载** | `model.save()` 保存全部 | `torch.save(state_dict)` 仅保存权重 | PyTorch架构与权重分离 |
| **训练/推理模式** | `call(inputs, training=True)` | `self.training` 属性 + `model.train()/eval()` | PyTorch自动管理，无需传参 |
| **正则化** | `keras.regularizers` 内置 | 手动在损失中添加 | PyTorch更透明 |
| **初始化器** | 类实例，返回张量 | `torch.nn.init` 函数，原地修改 | PyTorch是函数式，TF是面向对象 |

### 训练循环对比（核心差异）

```python
# TensorFlow - 需要GradientTape上下文管理器
with tf.GradientTape() as tape:
    y_pred = model(X_batch, training=True)
    loss = loss_fn(y_batch, y_pred)
gradients = tape.gradient(loss, model.trainable_variables)
optimizer.apply_gradients(zip(gradients, model.trainable_variables))

# PyTorch - 更自然，无需上下文管理器
y_pred = model(X_batch)
loss = loss_fn(y_pred, y_batch)
optimizer.zero_grad()    # 清零梯度（PyTorch默认累积梯度）
loss.backward()          # 自动计算梯度
optimizer.step()         # 更新参数
```

### 关键哲学差异

1. **PyTorch是"define-by-run"** — 计算图在运行时动态构建，调试更方便
2. **TF是"define-and-run"** — 计算图先定义后运行（TF2默认eager，但GradientTape仍保留了图的概念）
3. **PyTorch的`nn.Module`统一了层、模型和损失函数** — 它们都是同一个基类
4. **PyTorch的`self.training`属性** — 自动由`train()/eval()`管理，无需手动传递`training`参数

## 知识点总结

### 自定义组件速查表

| 组件 | 继承类 | 必须实现的方法 |
|-----|-------|---------------|
| 损失函数 | `nn.Module` 或简单函数 | `forward(y_pred, y_true)` |
| 评估指标 | 简单Python类 | `update()`, `compute()`, `reset()` |
| 层 | `nn.Module` | `__init__()` + `forward()` |
| 模型 | `nn.Module` | `__init__()` + `forward()` |
| 正则化 | 简单函数 | 在损失中手动添加 |

### 关键要点

1. **函数式组件**：简单场景使用普通函数即可
2. **类式组件**：需要保存参数时继承`nn.Module`并实现`forward()`
3. **参数定义**：在`__init__`中定义，PyTorch自动延迟初始化
4. **自定义训练循环**：`zero_grad()` -> `backward()` -> `step()` 三步曲
5. **保存/加载**：推荐保存`state_dict`，架构与权重分离

## 练习

### 练习1：实现带温度参数的Focal Loss

Focal Loss用于处理类别不平衡问题，公式为：

$$FL(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

请实现一个PyTorch版本的Focal Loss，要求：
- 继承`nn.Module`
- 支持可配置的`alpha`和`gamma`参数
- 支持二分类和多分类
- 在模拟数据上测试效果

### 练习2：实现自定义DropConnect层

DropConnect与Dropout类似，但DropConnect是随机丢弃权重而非激活值。

请实现一个`DropConnectLinear`层，要求：
- 继承`nn.Module`
- 训练时以概率p随机将权重置零
- 推理时使用缩放后的权重（类似Dropout的推理补偿）
- 与标准`nn.Linear`对比训练效果

### 练习3：实现完整的学习率调度器

请实现一个余弦退火学习率调度器（Cosine Annealing with Warm Restarts），要求：
- 在训练循环中动态调整学习率
- 支持热重启（周期性重置学习率）
- 可视化学习率变化曲线
- 对比固定学习率和余弦退火的效果

提示：PyTorch内置了`torch.optim.lr_scheduler.CosineAnnealingWarmRestarts`，可以先尝试自己实现，再与内置版本对比。